# Quotation Win/Loss Prediction — EDA + Logistic Regression Baseline

**Goal:** predict whether a construction quotation will be Won (`Success`=1) or Lost (`Success`=0), with an emphasis on explainability (coefficients, odds ratios, SHAP).

**This notebook covers:**
1. Load & inspect the data
2. EDA — missing values, target balance, leakage check, distributions, trends


In [1]:
# Libraries
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100


## Configuration
All the knobs that shape the pipeline live here so you can tune them without hunting through the notebook.

In [3]:
# Data path
DATA_PATH = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data.xlsx"

RANDOM_STATE = 42
TEST_FRACTION = 0.20        # last 20% of quotations by date held out as the test set
MIN_CATEGORY_COUNT = 15     # categories seen fewer times than this get grouped into "infrequent"
RARE_WALLTYPE_PCT = 1.0     # wall-type flags present in fewer than this % of rows get grouped

WALL_TYPE_COLS = ['Timber_RW','RC_Pile','Steel_Beam','Sheetpile','Anchor','Block','Shotcrete',
                  'Capping_Beam','Earthwork','Concrete_Slab','Precast','Culvert','Slip_Repair',
                  'Soil_Nail','Rock_RW','Bridge','Concrete','Design_and_Build','Budget','Drill_Only',
                  'Labour_Only','Driven_Pile','Palisade','Boardwalk','Soldier','Insitu','Barrier',
                  'Noise_RW','Base','Casing','Crib','DayWork','Flood_Repair','Micro_Pile','Reno',
                  'Temp_RW','Other']


## Load & Inspect

In [ ]:
# Load data
df_raw = pd.read_excel(DATA_PATH, sheet_name='Data')
print('Shape:', df_raw.shape)
#df_raw.head(9)

Shape: (4620, 55)


In [ ]:
# Explore missing value (naive) and uniqueness

info = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'n_missing (naive)': df_raw.isna().sum(),
    'pct_missing (naive)': (df_raw.isna().sum() / len(df_raw) * 100).round(1),
    'n_unique': df_raw.nunique(),
})
info


### ⚠️ Missing values are understated above

Several columns encode "missing" as a literal string (`"MISSING"`, `"Missing"`, `"na"`, `"unknown"`, blank spaces...) instead of a real `NaN`. Pandas' `.isna()` doesn't catch these, so the table above significantly *understates* real missingness for some columns. The next cell scans every text column for these sentinel tokens and converts them to proper `NaN` before we do anything else.

In [ ]:
# Explore the actual missing values
SENTINELS = {'missing', 'n/a', 'na', 'unknown', 'none', 'null', '-', 'tbc', 'tbd', '?', '', '00:00:00'}

df = df_raw.copy()
# include='string' matters here: pandas 3.x infers a dedicated StringDtype for text columns
# (shows as dtype 'str'), which select_dtypes(include='object') alone will silently miss.
text_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

sentinel_hits = {}
for c in text_cols:
    s = df[c].astype('string').str.strip()
    mask = s.str.lower().isin(SENTINELS)
    n = int(mask.sum())
    if n > 0:
        sentinel_hits[c] = n
        df.loc[mask, c] = pd.NA

comparison = pd.DataFrame({
    'pct_missing_naive': (df_raw.isna().sum() / len(df_raw) * 100).round(1),
    'pct_missing_actual': (df.isna().sum() / len(df) * 100).round(1),
})
comparison['difference'] = (comparison['pct_missing_actual'] - comparison['pct_missing_naive']).round(1)
comparison[comparison['difference'] > 0].sort_values('difference', ascending=False)


**Takeaway:** `Suburb` (~38%), `Priced_By` (~38%), `Date_Sent` (~48%) and `Due_Date` (~65%, plus placeholder text like `"ASAP"`/`"done"`) are far more incomplete than the naive check suggested. We'll handle each deliberately below rather than pretend they're clean.

### Target distribution (`Success`)

In [ ]:
# Explore the Target Distribution

target_counts = df['Success'].value_counts().sort_index()
target_pct = df['Success'].value_counts(normalize=True).sort_index() * 100
print(target_counts)
print((target_pct.round(1).astype(str) + '%').to_dict())

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=['Loss (0)', 'Won (1)'], y=target_counts.values, ax=ax, palette=['#c0392b', '#27ae60'])
ax.set_title('Target distribution')
ax.set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 30, f'{v}\n({target_pct.values[i]:.1f}%)', ha='center')
plt.tight_layout()
plt.show()


**~26% win rate overall.** This is moderate imbalance, not extreme — worth using `class_weight='balanced'` and looking at precision/recall/PR-AUC rather than raw accuracy, but not severe enough to need SMOTE-type resampling. Keep the naive "always predict Loss" baseline (~74% accuracy) in mind — a model needs to clearly beat that on the metrics that matter, not just on accuracy.

### Checking `Successful` and `number_of_successful` for target leakage

These two column names are suspiciously close to the target `Success`, and one is 99.6% missing despite the name implying it should be a count. Worth checking before they go anywhere near a model.

In [ ]:
# Check the useless features
print('=== Successful: sample non-null values ===')
print(df['Successful'].dropna().unique()[:10])
print()
print('=== Successful populated? vs Success ===')
print(pd.crosstab(df['Successful'].notna(), df['Success'], rownames=['Successful is populated'], colnames=['Success']))

print()
print('=== number_of_successful: sample non-null values ===')
print(df['number_of_successful'].dropna().unique()[:10])
print()
print('=== number_of_successful populated? vs Success ===')
print(pd.crosstab(df['number_of_successful'].notna(), df['Success'], rownames=['number_of_successful is populated'], colnames=['Success']))


**Both are being dropped, for different reasons:**

- **`Successful` is target leakage.** Its non-null values look like dollar amounts, and it is populated in 1195/1198 cases *exactly when* `Success`=1 — it appears to be the won contract value, recorded only after the outcome is known. Using it would let the model "predict" the outcome from a field that only exists because the outcome happened.
- **`number_of_successful` is mislabeled, not a leakage risk.** Despite the name, its values are free-text notes (`"lost by 20%"`, `"REQUESTED AREAS"`, ...), not counts, and it's 99.6% empty. It looks like this column and `Successful` may have gotten swapped or corrupted upstream in whatever process exports this report — worth a look at the source system if that's easy to check, since it could affect other reports too. Either way, not usable as a numeric feature.

**Please sanity-check the `Successful` interpretation against what you know of the source system** — if I've misread it, let me know before we move on, since dropping a real feature by mistake is worse than double-checking now.

### `Value` distribution

In [ ]:
# Explore the Value (numeric) column distribution
print(df['Value'].describe())
print()
print(f"Rows with Value <= 0: {(df['Value'] <= 0).sum()} ({(df['Value'] <= 0).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['Value'], bins=60, ax=axes[0], color='#2980b9')
axes[0].set_title('Value (raw)')
axes[0].set_xlabel('Value ($)')

sns.histplot(np.log1p(df['Value'].clip(lower=0)), bins=60, ax=axes[1], color='#8e44ad')
axes[1].set_title('Value (log1p transformed)')
axes[1].set_xlabel('log(1 + Value)')
plt.tight_layout()
plt.show()


`Value` ranges from \$0 to \$29.3M and is heavily right-skewed (mean \$331K, median only \$90K) — typical for project value data, but it would dominate a linear model and violate the roughly-linear-in-log-odds assumption if used raw. We'll use `log1p(Value)` as the model feature.

**Decision point:** 241 rows (5.2%) have `Value` ≤ 0. Their win rate is notably lower (10.4% vs 25.8% overall) and they skew toward `Design_and_Build`/`Budget`/`DayWork` project types — plausibly legitimate quotes without a fixed price at this stage, rather than data errors. **I'm keeping them** (log1p(0)=0 handles zero cleanly) rather than dropping ~5% of the data, but flagging this in case you know these should be treated differently (e.g. excluded, or a different value should be imputed).

### Win rate over time

In [ ]:
# Explore the Target Class(1) over Year
df['Date_parsed'] = pd.to_datetime(df['Date'], errors='coerce')
yearly = df.dropna(subset=['Date_parsed']).groupby(df['Date_parsed'].dt.year)['Success'].agg(['mean', 'count'])
yearly.index.name = 'Year'
print(yearly.round(3))

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(yearly.index, yearly['mean']*100, marker='o', color='#c0392b', linewidth=2)
ax1.set_ylabel('Win rate (%)', color='#c0392b')
ax1.set_xlabel('Year')
ax1.set_title('Win rate by year (bar = volume, line = win rate)')

ax2 = ax1.twinx()
ax2.bar(yearly.index, yearly['count'], alpha=0.25, color='#7f8c8d')
ax2.set_ylabel('Number of quotations', color='#7f8c8d')
ax2.grid(False)
plt.tight_layout()
plt.show()


**This is the single most important EDA finding.** Win rate held around 28–33% from 2013–2021, then dropped sharply: 21.9% (2023) → 16.2% (2024) → 19.0% (2025) → 10.2% (2026, partial year). It's roughly halved in the last three years relative to the historical average.

This has two implications:
1. **A random train/test split would be misleadingly optimistic** — it would mix easier historical patterns into the test set. We'll use a time-based split so the evaluation reflects predicting genuinely *future* quotations from *past* ones (confirmed quantitatively in Part 3).
2. **This is worth a conversation with the business**, not just a modeling footnote — do you know what changed around 2022–2023 (more competitors, pricing strategy, project mix, a specific client relationship)? That context could point to features we're missing entirely, and it affects how much to trust older rows as representative of current conditions.

### Seasonality within the year

In [ ]:
# Explore the Target Class(1) over Month
monthly = df.dropna(subset=['Date_parsed']).groupby(df['Date_parsed'].dt.month)['Success'].agg(['mean', 'count'])
monthly.index.name = 'Month'
print(monthly.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly.index, monthly['mean']*100, marker='o', color='#d35400', linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xlabel('Month')
ax.set_ylabel('Win rate (%)')
ax.set_title('Win rate by month, all years pooled')
ax.axhline(df['Success'].mean()*100, linestyle='--', color='gray', linewidth=1, label='Overall average')
ax.legend()
plt.tight_layout()
plt.show()


Win rate dips in Feb–Mar (~21%) and rises through to Sep–Oct–Dec (~29%) — roughly an 8-point swing, and importantly a **non-monotonic, wave-shaped** pattern rather than a steady rise or fall across the year. That shape is exactly what a single linear `Month` feature (1→12) cannot represent: a linear coefficient can only push predictions consistently up or down as the number increases, never both. This is the real justification for cyclical encoding — not just "December should be near January," but "the model needs two degrees of freedom to trace a curve that goes down, up, down again." Implemented in Part 2 as `Month_Sin`/`Month_Cos`, replacing the raw `Quote_Month` integer.

I checked day-of-month the same way (not plotted here) and found no comparable pattern — win rate by day-of-month is flat within noise (~25–27% regardless of early/mid/late month), and there's no obvious business mechanism for why the day within a month would affect a construction bid outcome. Adding `Day_Sin`/`Day_Cos` on top would mean two more features for a linear model already carrying ~135 encoded columns on 3,561 training rows, fit to what looks like pure noise. **Recommend skipping it** — happy to add it if you'd rather let the bootstrap significance check in Part 7 make that call empirically instead of me pre-judging it.

### `Priced_By`: quotations with more than one estimator

In [ ]:
# Explore the Categorical features which supposed to have 01 unique value but have multiple value 
priced_by_known = df[df['Priced_By'].notna()].copy()
priced_by_known['n_estimators'] = priced_by_known['Priced_By'].astype(str).str.split('/').apply(lambda parts: len([p for p in parts if p.strip() != '']))

print(priced_by_known['n_estimators'].value_counts().sort_index())
print()
print(priced_by_known.groupby(priced_by_known['n_estimators'] > 1)['Success'].agg(['mean', 'count']))
print()
print('Median Value, single vs multiple estimators:')
print(priced_by_known.groupby(priced_by_known['n_estimators'] > 1)['Value'].median())


Only ~1.7% of quotations (50 of 2880 with a known estimator) have more than one person credited, but it's a meaningfully different group: **46.0% win rate vs 32.3% for single-estimator quotes**, and a **4x higher median project value** (\$423,768 vs \$105,866). This lines up with what you'd expect operationally — bigger, more significant projects get more than one estimator's eyes on the price.

Worth building as its own `Multi_Estimator` flag rather than relying on the existing one-hot of `Priced_By` combos: at `MIN_CATEGORY_COUNT=15`, almost every specific combo (`"GARY / SHERYL"`, `"GARY / ZEINAB"`, ...) falls below the threshold and gets grouped into the generic "infrequent" bucket alongside rare *individual* estimator names — which conflates two different things ("rarely-seen person" and "co-estimated project") and throws away exactly the pattern found above. An explicit flag keeps it intact. One caveat: since it's correlated with `Value_Log` (already in the model), some of its apparent effect may overlap with "this is just a large project" rather than being fully independent — worth watching in the coefficient/SHAP results in Part 7 rather than assuming it's a clean standalone driver.

### `Due_Date` and `Date_Sent`: are they trustworthy?

Flagged for a closer look: rows where a date shows up as an epoch-style placeholder (e.g. `0/01/1900`, the classic "Excel serial day 0" artifact) rather than a real missing value pandas would already catch. Checked directly against this file:

In [ ]:
# Check the missing / invalid values in Date columns
for c in ['Due_Date', 'Date_Sent']:
    parsed = pd.to_datetime(df[c], errors='coerce')
    epoch_like = parsed.dt.year <= 1900
    raw_types = df[c].map(lambda x: type(x).__name__).value_counts().to_dict()
    print(f'{c}: parsed range {parsed.min()} -> {parsed.max()}')
    print(f'  rows with parsed year <= 1900: {epoch_like.sum()}')
    print(f'  raw python types in this column: {raw_types}')
    print()


No epoch-style dates turn up in this file as read by pandas — every cell is either a genuine `datetime` object or the literal text `"MISSING"` (plus the handful of `"ASAP"`/`"done"` placeholders found earlier), and there are zero raw unconverted serial numbers. My best guess for the discrepancy: if you're viewing this file directly in Excel (or another BI/spreadsheet tool), some tools render text that can't be parsed as a date — like `"MISSING"` sitting in a date-formatted column — as the epoch (`0/01/1900`) rather than blank. If that's not what's going on, send me a specific `JOB NO` where you're seeing it and I'll dig further — but either way, this doesn't change the bottom line, which the next check gets at more directly.

In [ ]:
# Explore usable rate in Date columns
d = pd.to_datetime(df['Date'], errors='coerce')
dd = pd.to_datetime(df['Due_Date'], errors='coerce')
ds_ = pd.to_datetime(df['Date_Sent'], errors='coerce')
df['due_valid'] = dd.notna() & ((dd - d).dt.days >= 0)
df['sent_valid'] = ds_.notna() & ((ds_ - d).dt.days >= 0)

print('=== % of rows with a usable Due_Date, by year ===')
print(df.groupby(d.dt.year)['due_valid'].mean().round(3))
print()
print('=== % of rows with a usable Date_Sent, by year ===')
print(df.groupby(d.dt.year)['sent_valid'].mean().round(3))


**Here's the real issue, and it's the same one found for `Priced_By`:** both fields go from essentially 0% populated before 2020 to mostly populated from 2020 onward — practically a step function, not a random gap. That means a `Due_Date`/`Date_Sent`-missing flag is mostly standing in for "pre-2020 record" a third time over, on top of `Quote_Year` and `Priced_By_Unknown_Estimator` already carrying that signal. Tested directly: dropping `Due_Date`/`Date_Sent` (and everything derived from them) entirely and comparing against keeping them —

| | ROC-AUC | PR-AUC | Encoded features |
|---|---|---|---|
| With `Lead_Time_Days`/`Turnaround_Days` + missing flags | 0.597 | 0.256 | 137 |
| Without (Date-derived features only) | 0.616 | 0.254 | 132 |

No performance cost — if anything, slightly better — for five fewer, less confounded features. **Dropping `Due_Date` and `Date_Sent` entirely**, keeping only `Date` (via `Quote_Year`, `Month_Sin`, `Month_Cos`) to carry the time signal.

### Categorical cardinality

In [ ]:
# Explore the categorical cardinality

cat_cols = ['CLIENT', 'Client_Clean', 'ADDRESS', 'Suburb', 'CONTACTS', 'Contact_Clean', 'Priced_By', 'DESCRIPTION']
card = pd.DataFrame({
    'n_unique': df[cat_cols].nunique(),
    'pct_missing': (df[cat_cols].isna().sum() / len(df) * 100).round(1)
}).sort_values('n_unique', ascending=False)
card


`ADDRESS` (4539 unique) and `CONTACTS`/`Contact_Clean` (~2100–2900 unique) are essentially free text / near-identifiers — not usable as categorical features directly, and each already has a cleaner counterpart (`Suburb`, and `Client_Clean` for the client relationship). `Client_Clean` (808 unique) and `Suburb` (213 unique) are the two high-cardinality fields worth keeping, with rare-category grouping (below) to keep them usable in a linear model.

### Wall/work-type binary flags (37 columns)

In [ ]:
# Explore the prevalance of the Binary features and their relationship to the Target Class(1)

prevalence = (df[WALL_TYPE_COLS].mean() * 100).sort_values(ascending=False)
success_rate = pd.Series({c: df.loc[df[c]==1, 'Success'].mean() for c in WALL_TYPE_COLS})
n_when_1 = df[WALL_TYPE_COLS].sum()

flag_summary = pd.DataFrame({'prevalence_pct': prevalence, 'n_rows': n_when_1, 'success_rate_when_1': success_rate.round(3)})
flag_summary = flag_summary.sort_values('prevalence_pct', ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
sns.barplot(x=flag_summary['prevalence_pct'], y=flag_summary.index, ax=ax, color='#16a085')
ax.axvline(RARE_WALLTYPE_PCT, color='#c0392b', linestyle='--', label=f'{RARE_WALLTYPE_PCT}% grouping threshold')
ax.set_xlabel('% of quotations')
ax.legend()
ax.set_title('Work-type flag prevalence')
plt.tight_layout()
plt.show()

flag_summary


22 of the 37 flags appear in under 1% of rows — several with fewer than 10 occurrences total (e.g. `Earthwork`: 4 rows, `Micro_Pile`: 9 rows). A few even show 0% or 100% success rate purely because the sample is tiny (`DayWork`: 16/16 won, `Micro_Pile`: 0/9 won) — real patterns or noise, impossible to tell at that sample size. Giving each of these its own coefficient in a linear model would fit noise, not signal, so **rare flags (<1% prevalence) get grouped into a single `Rare_WorkType` indicator**, keeping the 15 flags with enough data to estimate reliably as individual features. Tree-based models later can likely handle the raw 37 flags fine — we can revisit this when we get there.

### Correlation among numeric/binary features

In [ ]:
# Explore the Correlation among the numeric/binary features

corr_cols = ['Value', 'No_Wall_Types'] + list(flag_summary[flag_summary['prevalence_pct']>=1.0].index) + ['Success']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False, ax=ax, square=True, cbar_kws={'shrink': 0.7})
ax.set_title('Correlation matrix — numeric & common work-type flags')
plt.tight_layout()
plt.show()


No alarming pairwise correlations among the common flags — nothing suggesting we need to drop one of a redundant pair before modeling.